# RSNA knee: submission smoke test
Constant probabilities check the notebook execution path; this is not a trained image model.
Attach the competition data, use CPU, and disable internet. Reads test IDs at runtime and does not use reports.
Local execution is supported from the repository root or notebooks directory.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the mounted competition at runtime; never hard-code example study IDs.
roots = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("data/raw"), Path("../data/raw"),
]
data_root = next((p for p in roots if (p / "test.csv").is_file()), None)
if data_root is None:
    raise FileNotFoundError("Attach the RSNA knee competition data or download local metadata.")

test = pd.read_csv(data_root / "test.csv")
sample = pd.read_csv(data_root / "sample_submission.csv")
id_column = "StudyInstanceUID"
targets = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
if list(sample.columns) != [id_column, *targets]:
    raise ValueError("Unexpected sample submission schema")
ids = test[id_column]
if test.empty or ids.isna().any() or ids.duplicated().any() or ids.astype(str).str.strip().eq("").any():
    raise ValueError("Invalid test IDs")
submission = test[[id_column]].copy()
submission[targets] = 0.5
values = submission[targets].to_numpy()
assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()

if Path("/kaggle/working").is_dir():
    output_dir = Path("/kaggle/working")
else:
    output_dir = data_root.resolve().parent.parent / "artifacts/baselines/notebook_smoke"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "submission.csv"
submission.to_csv(output_path, index=False)
print(f"Wrote {len(submission)} studies to {output_path}. Constant smoke test only.")
